# Week 3 — Synthetic Dataset Generator

- **Write models that can generate a dataset:** LLM generates synthetic rows (e.g. JSON/CSV) from your description.
- **Use a variety of models for diverse output:** Choose one model or "Multiple models" to generate different portions with Gemini, GPT-4, and Claude.
- **Gradio UI:** Describe your data, set number of rows and format, pick model(s), generate and download.

In [1]:
import os
import json
import csv
import io
import tempfile
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

load_dotenv()

# OpenRouter client (same key as Week 1 & 2: OPENROUTER_API_KEY)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# Variety of models for diverse output
MODELS = [
    "google/gemini-2.0-flash-001",
    "openai/gpt-4o",
    "anthropic/claude-3.5-sonnet",
]
MULTI_LABEL = "Multiple models (diverse)"

SYSTEM_PROMPT = """You are a synthetic data generator. Given a description of a dataset, you output only valid data rows—no extra commentary.

If the user asks for JSON: output a JSON array of objects, one object per row, with the exact keys they specify. No markdown, no explanation.
If the user asks for CSV: output CSV with a header row, then one row per record. No markdown, no explanation.

Be consistent with the schema. Generate realistic, varied values."""

def generate_dataset(description, num_rows, output_format, model_choice):
    if not (description or "").strip():
        return "Please describe the dataset you want (e.g. columns and type of data).", None
    num_rows = max(1, min(int(num_rows or 10), 100))
    fmt = (output_format or "JSON").strip().upper()
    if fmt not in ("JSON", "CSV"):
        fmt = "JSON"

    if model_choice == MULTI_LABEL:
        # Use a variety of models for diverse output: split work across models
        models_to_use = MODELS
        rows_per_model = max(1, num_rows // len(models_to_use))
        remainder = num_rows - rows_per_model * len(models_to_use)
    else:
        models_to_use = [model_choice]
        rows_per_model = num_rows
        remainder = 0

    user_prompt = f"""Generate exactly {num_rows} synthetic data rows.

Dataset description:
{description.strip()}

Output format: {fmt}
Output only the {fmt} data, nothing else."""

    all_parts = []
    for i, model in enumerate(models_to_use):
        n = rows_per_model + (1 if i < remainder else 0)
        if n <= 0:
            continue
        prompt = f"""Generate exactly {n} synthetic data rows.

Dataset description:
{description.strip()}

Output format: {fmt}
Output only the {fmt} data, nothing else."""
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                stream=False,
                extra_headers={"HTTP-Referer": "http://localhost:3000", "X-Title": "Synthetic Data Generator"},
            )
            text = (resp.choices[0].message.content or "").strip()
            # Strip markdown code blocks if present
            if text.startswith("```"):
                lines = text.split("\n")
                if lines[0].startswith("```"):
                    lines = lines[1:]
                if lines and lines[-1].strip() == "```":
                    lines = lines[:-1]
                text = "\n".join(lines)
            all_parts.append(text)
        except Exception as e:
            all_parts.append(f"Error with {model}: {e}")

    combined = "\n".join(p for p in all_parts if p)
    if not combined:
        return "No data generated. Check your API key and description.", None

    # Build downloadable file
    if fmt == "JSON":
        try:
            # Try to parse and re-serialize for valid JSON
            if combined.strip().startswith("["):
                data = json.loads(combined)
            else:
                data = [json.loads(line) for line in combined.strip().split("\n") if line.strip()]
            file_content = json.dumps(data, indent=2)
        except Exception:
            file_content = combined
        filename = "synthetic_data.json"
    else:
        file_content = combined
        filename = "synthetic_data.csv"

    display = f"**Format:** {fmt}\n\n**Preview:**\n\n```\n{combined[:3000]}{'...' if len(combined) > 3000 else ''}\n```"
    # Gradio File needs a path; write to temp file
    suffix = ".json" if fmt == "JSON" else ".csv"
    with tempfile.NamedTemporaryFile(mode="w", suffix=suffix, delete=False, encoding="utf-8") as tmp:
        tmp.write(file_content)
        file_path = tmp.name
    return display, file_path

# Gradio UI
model_choices = [MULTI_LABEL] + MODELS
with gr.Blocks(title="Synthetic Dataset Generator") as demo:
    gr.Markdown("## Generate a synthetic dataset with one or many models")
    description = gr.Textbox(
        label="Dataset description",
        placeholder="e.g. 10 rows: product_name (string), rating (1-5), review_text (string), date (YYYY-MM-DD). Product: wireless earbuds.",
        lines=4,
    )
    with gr.Row():
        num_rows = gr.Number(label="Number of rows", value=10, minimum=1, maximum=100)
        output_format = gr.Radio(label="Output format", choices=["JSON", "CSV"], value="JSON")
        model_selector = gr.Dropdown(label="Model", choices=model_choices, value=MODELS[0])
    btn = gr.Button("Generate dataset")
    out_text = gr.Markdown(label="Output")
    out_file = gr.File(label="Download")

    btn.click(
        fn=generate_dataset,
        inputs=[description, num_rows, output_format, model_selector],
        outputs=[out_text, out_file],
    )

demo.launch(inline=True)

C:\Users\kwx1494728\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
